In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/zomato.csv')
print(df.shape)
print(df.dtypes)
df.head()

: 

In [ ]:
# Drop irrelevant columns
df.drop(columns=['url', 'address', 'phone', 'reviews_list', 'menu_item'], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Clean 'rate' column: "4.1/5" -> 4.1, handle "NEW" and "-"
df['rate'] = df['rate'].astype(str).str.replace('/5', '').str.strip()
df['rate'] = pd.to_numeric(df['rate'], errors='coerce')

# Clean cost column
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str).str.replace(',', '').str.strip()
df['approx_cost(for two people)'] = pd.to_numeric(df['approx_cost(for two people)'], errors='coerce')

# Rename for convenience
df.rename(columns={
    'approx_cost(for two people)': 'cost',
    'listed_in(type)': 'listing_type',
    'listed_in(city)': 'listed_city'
}, inplace=True)

# Binary encode
df['online_order'] = (df['online_order'] == 'Yes').astype(int)
df['book_table'] = (df['book_table'] == 'Yes').astype(int)

# Drop rows with null rate or cost (needed for model)
df.dropna(subset=['rate', 'cost', 'location', 'cuisines', 'rest_type'], inplace=True)

print(df.shape)
print(df.isnull().sum())
df.head()

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Rating Distribution', 'Cost Distribution',
                    'Votes Distribution', 'Online Order vs Book Table'))

# Rating
fig.add_trace(go.Histogram(x=df['rate'], nbinsx=30, name='Rating',
    marker_color='#FF6B6B'), row=1, col=1)

# Cost
fig.add_trace(go.Histogram(x=df['cost'], nbinsx=40, name='Cost',
    marker_color='#4ECDC4'), row=1, col=2)

# Votes (log scale)
fig.add_trace(go.Histogram(x=np.log1p(df['votes']), nbinsx=30, name='log(Votes)',
    marker_color='#45B7D1'), row=2, col=1)

# Online order vs Book table
categories = ['Online Order', 'Book Table']
yes_counts = [df['online_order'].sum(), df['book_table'].sum()]
no_counts = [len(df) - v for v in yes_counts]
fig.add_trace(go.Bar(name='Yes', x=categories, y=yes_counts, marker_color='#96CEB4'), row=2, col=2)
fig.add_trace(go.Bar(name='No', x=categories, y=no_counts, marker_color='#FFEAA7'), row=2, col=2)

fig.update_layout(height=700, title_text='<b>Basic Distributions</b>',
    title_font_size=20, showlegend=False, barmode='stack')
fig.show()

# Key stats
print(f"Avg Rating: {df['rate'].mean():.2f}")
print(f"Median Cost for Two: ₹{df['cost'].median():.0f}")
print(f"Online Order Adoption: {df['online_order'].mean()*100:.1f}%")
print(f"Table Booking Adoption: {df['book_table'].mean()*100:.1f}%")

Observations
- Ratings are normally distributed around 3.7 — not many truly bad or truly exceptional restaurants
- Cost is right-skewed — majority of restaurants are budget (₹300–800), long tail of premium
- 65.7% online ordering but only 15.2% table booking — huge gap, interesting business insight
- Votes are log-normal — a few restaurants dominate engagement

In [ ]:
# Top 15 locations by restaurant count
top_locations = df['location'].value_counts().head(15)

fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Top 15 Locations by Count', 'Avg Rating by Location (Top 15)',
                    'Restaurant Type Distribution', 'Avg Cost by Restaurant Type'))

# Location count
fig.add_trace(go.Bar(x=top_locations.values, y=top_locations.index,
    orientation='h', marker_color='#FF6B6B', name='Count'), row=1, col=1)

# Avg rating by location
avg_rating_loc = df.groupby('location')['rate'].mean().sort_values(ascending=False).head(15)
fig.add_trace(go.Bar(x=avg_rating_loc.values, y=avg_rating_loc.index,
    orientation='h', marker_color='#4ECDC4', name='Avg Rating'), row=1, col=2)

# Restaurant type
rest_type_counts = df['rest_type'].str.split(',').explode().str.strip().value_counts().head(10)
fig.add_trace(go.Bar(x=rest_type_counts.index, y=rest_type_counts.values,
    marker_color='#45B7D1', name='Rest Type'), row=2, col=1)

# Avg cost by restaurant type
avg_cost_type = df.groupby('rest_type')['cost'].mean().sort_values(ascending=False).head(10)
fig.add_trace(go.Bar(x=avg_cost_type.index, y=avg_cost_type.values,
    marker_color='#96CEB4', name='Avg Cost'), row=2, col=2)

fig.update_layout(height=750, title_text='<b>Location & Restaurant Type Analysis</b>',
    title_font_size=20, showlegend=False)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_xaxes(tickangle=45, row=2, col=2)
fig.show()

print("Top 5 locations by restaurant count:")
print(top_locations.head())
print("\nTop 5 locations by avg rating:")
print(avg_rating_loc.head())

Insights:
- BTM dominates in count (3,873) but doesn't appear in top rated — volume ≠ quality
- Lavelle Road has highest avg rating (4.14) — premium locality effect
- Quick Bites is the most common type but Fine Dining commands highest cost (₹2,500-3,000)
- Top rated locations are all central/upmarket — Church Street, Lavelle Road, St. Marks Road

In [ ]:
# Top cuisines
top_cuisines = df['cuisines'].str.split(',').explode().str.strip().value_counts().head(15)

# Cost-Rating correlation by location
avg_by_loc = df.groupby('location').agg({'rate':'mean','cost':'mean','votes':'sum'}).reset_index()
avg_by_loc = avg_by_loc[avg_by_loc['votes'] > 1000]  # filter low-activity locations

fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Top 15 Cuisines', 'Cost vs Rating (Bubble = Total Votes)',
                    'Online Order Impact on Rating', 'Table Booking Impact on Rating'))

# Top cuisines
fig.add_trace(go.Bar(x=top_cuisines.values, y=top_cuisines.index,
    orientation='h', marker_color='#FF6B6B'), row=1, col=1)

# Bubble chart: cost vs rating
fig.add_trace(go.Scatter(
    x=avg_by_loc['cost'], y=avg_by_loc['rate'],
    mode='markers+text', text=avg_by_loc['location'],
    textposition='top center', textfont=dict(size=7),
    marker=dict(size=np.sqrt(avg_by_loc['votes'])/15,
                color=avg_by_loc['rate'], colorscale='RdYlGn',
                showscale=True)), row=1, col=2)

# Online order vs rating
online_rating = df.groupby('online_order')['rate'].mean()
fig.add_trace(go.Bar(
    x=['No Online Order', 'Has Online Order'],
    y=online_rating.values,
    marker_color=['#FF6B6B','#96CEB4']), row=2, col=1)

# Book table vs rating
book_rating = df.groupby('book_table')['rate'].mean()
fig.add_trace(go.Bar(
    x=['No Table Booking', 'Has Table Booking'],
    y=book_rating.values,
    marker_color=['#FF6B6B','#96CEB4']), row=2, col=2)

fig.update_layout(height=750, title_text='<b>Cuisine & Feature Impact Analysis</b>',
    title_font_size=20, showlegend=False)
fig.update_yaxes(range=[3.4, 4.5], row=2, col=1)
fig.update_yaxes(range=[3.4, 4.5], row=2, col=2)
fig.show()

print(f"Avg rating WITH online order: {df[df['online_order']==1]['rate'].mean():.3f}")
print(f"Avg rating WITHOUT online order: {df[df['online_order']==0]['rate'].mean():.3f}")
print(f"Avg rating WITH table booking: {df[df['book_table']==1]['rate'].mean():.3f}")
print(f"Avg rating WITHOUT table booking: {df[df['book_table']==0]['rate'].mean():.3f}")

Insights
- Table booking has a massive rating impact — 4.14 vs 3.62 (0.52 difference). This is your headline insight
- Online order difference is small (3.72 vs 3.66) — marginal effect
- North Indian dominates cuisines by far, followed by Chinese and South Indian
- Lavelle Road bubble is large AND green — high cost, high rating, high engagement. Premium sweet spot
- BTM is the small red dot bottom left — high volume, low rating, low cost

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create target: high_performer = rating >= 4.0 AND votes >= 200
df['high_performer'] = ((df['rate'] >= 4.0) & (df['votes'] >= 200)).astype(int)
print(f"High performers: {df['high_performer'].sum()} ({df['high_performer'].mean()*100:.1f}%)")

# Encode categoricals for correlation
df_encoded = df.copy()
le = LabelEncoder()
for col in ['location', 'rest_type', 'cuisines', 'listing_type', 'listed_city']:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# Correlation heatmap
numeric_cols = ['online_order','book_table','rate','votes','cost',
                'listing_type','high_performer']
corr = df_encoded[numeric_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.columns,
    colorscale='RdBu',
    zmid=0,
    text=np.round(corr.values, 2),
    texttemplate='%{text}',
    textfont=dict(size=11)))

fig.update_layout(height=500, title='<b>Feature Correlation Heatmap</b>',
    title_font_size=20)
fig.show()

# Class balance
print(f"\nClass distribution:")
print(df['high_performer'].value_counts())

Insights:

- rate (0.63) and votes (0.54) are strongest predictors of high_performer — expected since they define it
- book_table (0.50) is a strong independent signal — confirms our earlier finding
- cost (0.44) correlates too — pricier restaurants tend to perform better
- online_order (0.02) is nearly irrelevant — surprising and interview-worthy
- Class is imbalanced 78/22 — we'll handle this with class_weight in XGBoost

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

# Features
feature_cols = ['online_order', 'book_table', 'votes', 'cost',
                'location', 'rest_type', 'cuisines', 'listing_type']

X = df_encoded[feature_cols]
y = df_encoded['high_performer']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# XGBoost with class imbalance handling
scale_pos = (y==0).sum() / (y==1).sum()

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
import shap

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig = go.Figure(data=go.Heatmap(
    z=cm, x=['Predicted 0','Predicted 1'],
    y=['Actual 0','Actual 1'],
    colorscale='Blues',
    text=cm, texttemplate='%{text}',
    textfont=dict(size=16)))
fig.update_layout(title='<b>Confusion Matrix</b>', height=400)
fig.show()

# SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary plot
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols,
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance', fontsize=14)
plt.tight_layout()
plt.savefig('../shap_outputs/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Beeswarm
# Beeswarm with sample to avoid memory error
plt.figure(figsize=(10,6))
sample_idx = np.random.choice(len(X_test), 2000, replace=False)
shap.summary_plot(shap_values[sample_idx], X_test.iloc[sample_idx],
                  feature_names=feature_cols, show=False)
plt.title('SHAP Beeswarm Plot', fontsize=14)
plt.savefig('../shap_outputs/shap_beeswarm.png', dpi=100, bbox_inches='tight')
plt.show()

Insights

- votes dominates everything — 10x more important than any other feature. Low votes (blue) strongly push predictions negative; high votes (red) strongly push positive. This means engagement is the single biggest signal of restaurant success on Zomato.
- book_table — red dots (has booking) push right = positive impact. Confirms our earlier finding structurally.
- online_order — nearly flat around zero. SHAP confirms it adds almost no predictive value despite 65% adoption. Great counter-intuitive talking point.
- cost — mixed direction, meaning both very cheap and very expensive restaurants can succeed, but mid-range is uncertain.